# 4. Partie climat : retirer l'effet de la pluie

Hormis la géographie, il y a aussi le climat qui joue beaucoup sur les variations de niveau des nappes : le niveau monte quand il pleut et diminue quand le sol est sec et a besoin d'eau. Pour juger l'effet de l'industrie toutes choses égales par ailleurs, il faut donc aussi retirer les effets de la météo. 

Le résidu qui restera sera la partie que la pluie n'explique pas. 
Ce retrait reste une estimation car une partie de l'effet humain peut passer pour du climat.

Ce notebook retire l'effet de la pluie.

Variables produites : `pluie_eff_mensuelle`, `partie_climat`, `residu`, `tendances`, `resume_tendances`.

In [15]:
%%capture
#Étapes précédentes : exécutées sans afficher leurs figures ni leurs messages, et une seule fois par noyau
#(si les données ne sont pas encore en cache, exécuter d'abord eau_01_donnees.ipynb seul pour suivre le téléchargement)
AFFICHER = globals().get("AFFICHER", True)   #True si ce notebook est ouvert seul, False s'il est appelé par un autre notebook
_afficher_etape_04, AFFICHER = AFFICHER, False
if "ETAPE_03" not in globals():
    %run ./geographie.ipynb
AFFICHER = _afficher_etape_04
plt.close("all")

## 4.1 Pluie efficace SAFRAN

Météo-France publie en open data la réanalyse SAFRAN : une valeur par jour sur une grille de 8 km.
On garde la pluie efficace PE (pluie moins évaporation, en mm) de la maille la plus proche de chaque station.
On remonte à 2012 car une nappe a de la mémoire : elle réagit aux pluies du passé.

In [16]:
URL_SAFRAN = "https://meteofrance.s3.sbg.io.cloud.ovh.net/data/REF_CC/SIM/QUOT_SIM2_{}.csv.gz"   #Les {} seront remplacées par l'année
lien_pluie = "../../data/pluie_efficace_stations.csv.gz"   #Cache : une ligne par station et par mois

#SAFRAN est en Lambert II étendu (EPSG:27572), avec des coordonnées en hectomètres (x 100 pour repasser en mètres)
stations_lambert2 = (stations
                     .drop_duplicates(subset="code_bss")
                     .set_index("code_bss")
                     .to_crs(epsg=27572)
                     [["geometry"]])   #Doubles crochets : on garde un GeoDataFrame (et pas une simple série de points), sjoin_nearest en a besoin

#Téléchargement, seulement la première fois (environ 2 Go) : ensuite on relit le fichier local
if not os.path.exists(lien_pluie):
    # Code légèrement sous optimal pour une question de facilité de lecture du code et de compréhension : on telecharge 2 fois 2012 mais c'est minime
    # On prend le fichier des pluies et on extrait la grille, ensuite on fait l'appariement avec les stations de la partie précédente et 
    # enfin on garde seulement les mailles utiles. Après on fait des jointures entre les mailles utiles calculées qui nous intéresse et les pluies


    #Chaque maille apparaît une fois par jour dans le fichier, on ne garde que la liste des mailles
    grille = (pd.read_csv(URL_SAFRAN.format(2012), sep=";", usecols=["LAMBX", "LAMBY"])
                .rename(columns={"LAMBX": "X", "LAMBY": "Y"})
                .drop_duplicates())
    
    #Type nécessaire pour faire le sjoin_nearest et passage des hm en m
    grille = gpd.GeoDataFrame(grille, geometry=gpd.points_from_xy(grille["X"] * 100, grille["Y"] * 100), crs="EPSG:27572")
    
    appariement = gpd.sjoin_nearest(stations_lambert2, grille, distance_col="distance")

    appariement = appariement.reset_index().drop_duplicates(subset="code_bss")   #En cas d'égalité de distance, on garde une seule maille par station
    appariement = appariement[["code_bss", "X", "Y", "distance"]]     # On ne conserve que les colonnes qui nous intéressent
    
    #Les mailles utiles : plusieurs stations partagent souvent la même maille, on ne garde chaque maille qu'une fois,
    mailles_utiles = appariement[["X", "Y"]].drop_duplicates()


    def pluie_mensuelle(annee):
        """Pluie efficace mensuelle des mailles utiles pour une année (un fichier par an pour toute la France, environ 140 Mo)."""
        safran_annee = pd.read_csv(URL_SAFRAN.format(annee), sep=";", usecols=["LAMBX", "LAMBY", "DATE", "PE"]).rename(columns={"LAMBX": "X", "LAMBY": "Y"})

        #On ne garde que les lignes SAFRAN des mailles utiles pour réduire les coûts de calcul, c'est l'intérêt de mailles_utiles
        safran_annee = safran_annee.merge(mailles_utiles, how="inner", on=["X", "Y"])

        #Cumul mensuel de la pluie efficace par maille
        #SAFRAN écrit les dates sous la forme 20120101 ; on supprime le jour en enlevant les 2 derniers chiffres
        safran_annee["mois"] = safran_annee["DATE"]//100
        return safran_annee.groupby(["X", "Y", "mois"])["PE"].sum().reset_index()

    # On part de 2012 car le modèle utilise jusqu'à 36 mois de pluie avant 2015. On télécharge 4 années à la fois pour aller plus vite
    # L'étape lente est le téléchargement : le programme attend surtout le réseau, il ne calcule presque pas.
    # Les threads en profitent : pendant qu'un téléchargement attend, un autre avance.
    from concurrent.futures import ThreadPoolExecutor
    # Pool de 4 threads : au plus 4 années en même temps, assez pour gagner du temps sans saturer la mémoire (~140 Mo par fichier).
    # Le with ferme le pool proprement à la fin, après avoir attendu que tous les threads aient terminé.
    with ThreadPoolExecutor(max_workers=4) as executeur:
        # Même principe que map() : pluie_mensuelle est appelée sur chaque année de 2012 à 2025 (2026 exclu), chaque appel confié à un ouvrier libre.
        # Les résultats reviennent dans l'ordre des années ; list() attend la fin de tout et donne 14 DataFrames, recollés plus bas par pd.concat.
        morceaux = list(executeur.map(pluie_mensuelle, range(2012, 2026)))


    # On stocke tout dans un même fichier qui contient les colonnes : X, Y, date/mois, pluie efficace, code BSS des stations et distance (entre le centre de la grille et la station)
    # Il y a plein de lignes pour une même station ou un même mois
    # si on veut les appariements, on drop_duplicates sur la colonne code_bss en ne gardant que les colonnes X, Y, distance
    # si on veut la pluie efficace mensuelle, on fait .pivot en prenant les mois pour index, les stations pour colonnes et la pluie efficace comme valeur

    cache_csv = appariement.merge(pd.concat(morceaux), on=["X", "Y"])
    cache_csv["mois"] = pd.to_datetime(cache_csv["mois"].astype(str), format="%Y%m") #sans jour donné dans le str, pandas met le 1er du mois, ce qu'il nous faut pour la suite
    # Au prochain lancement, on relit ce petit fichier au lieu de retélécharger 2 Go
    cache_csv[["code_bss", "X", "Y", "distance", "mois", "PE"]].to_csv(lien_pluie, index=False, compression="gzip")

cache_csv = pd.read_csv(lien_pluie, parse_dates=["mois"])

pluie_eff_mensuelle = cache_csv.pivot(index="mois", columns="code_bss", values="PE")   #Une ligne par mois, une colonne par station
appariement = cache_csv.drop_duplicates(subset="code_bss").set_index("code_bss")[["X", "Y", "distance"]]   #Une ligne par station : sa maille

## 4.2 Modèle climat → nappe

Pour chaque station, on explique le niveau mensuel par une régression linéaire :

$$\text{niveau}_t = a + b \, PE_{k \text{ derniers mois}} + c \, PE_{3 \text{ derniers mois}} + e \, PE_{6 \text{ derniers mois}} + \text{saison (sinus, cosinus)} + d \, \text{temps}$$


On ajuste par station pour bien retirer ce qui est propre à la nappe, c'est-à-dire ses caractéristiques propres qui rentrent dans la constante, ainsi que sa localisation qui détermine la quantité de pluie qu'elle reçoit, grandeur que l'on essaie de retirer pour mieux observer ses variations. 

Comme vu à la partie précédente, les nappes ne sont pas toutes identiques en termes de caractéristiques : on essaie plusieurs valeurs de « mémoire » pour modéliser les différents types de roches et les différentes profondeurs (et bien sûr les différentes tailles de nappes que la partie précédente n'a pas pu mettre en lumière). On essaie k = 12, 24 et 36 mois et on garde le meilleur.

Le terme $d \times t$ ($t$ = temps en années) capte la tendance temporelle mais non climatique. Sans lui, la régression attribuerait à la pluie
toute la part de la baisse qui évolue en même temps qu'elle (années sèches puis recharge record de 2023-2024) et sous-estimerait la tendance.

On estime tout ensemble, puis on garde seulement $d \times t$ de la simulation : la partie_climat = constante + pluie + saison est retirée.

$a$ porte la géographie de la station : l'enlever du résidu permet de comparer les stations.

Le R² mesure la qualité du modèle complet (climat + tendance) : on garde le plus grand comme critère pour le choix de k.

La tendance est mesurée par la médiane des pentes entre les paires d'un même mois (janvier 2015 avec janvier 2020...).
On a pris ce parti car nous estimons que comparer deux mois différents mélangerait la tendance et le cycle saisonnier.

In [17]:
# Régression linéaire pour trouver, station par station, la part du niveau expliquée par la pluie
# On maximise le R² pour expliquer le plus possible les variations par le climat 
# de telle sorte que le résidu est expliqué presque entièrement par le reste : l'action humaine ou d'autres causes. 
def modele_climat(st):
    """Régression du niveau sur la pluie efficace cumulée et le temps ; renvoie (R², k, niveau simulé par le climat seul)."""
    pe, y = pluie_eff_mensuelle[st], niveaux_mensuels[st] 

    meilleur = (-np.inf, None, None) #initialisation du triplet recherché

    for k in [12, 24, 36]:   #k = 6 ferait doublon avec pe_6
        X = pd.DataFrame({"pe_k": pe.rolling(k).sum(), "pe_3": pe.rolling(3).sum(), "pe_6": pe.rolling(6).sum()}).reindex(MOIS)
        # on définit un dataframe pour notre régression qu'on initialise avec la pluie efficace des derniers mois en testant plusieurs combinaisons
        # grâce à des sommes glissantes et avec une reindexation après

        X["sin"], X["cos"], X["constante"] = np.sin(2 * np.pi * MOIS.month / 12), np.cos(2 * np.pi * MOIS.month / 12), 1.0 
        # on définit les variables pour la régression : sin(2*pi*n°mois) et cos(2*pi*n°mois) pour capter les non linéarités dues aux saisons 
        # et la constante pour le biais

        X["tendance"] = (MOIS - MOIS[0]).days / 365.25
        # temps en années depuis janvier 2015 : la tendance non climatique est estimée EN MÊME TEMPS que l'effet de la pluie,
        # sinon la régression attribue à la pluie la part de la baisse qui évolue avec elle (années sèches puis recharge 2023-2024)

        non_vide = y.notna()

        beta = np.linalg.lstsq(X[non_vide], y[non_vide], rcond=None)[0]
        complet = X @ beta
        # produit "matriciel" entre le df des variables explicatives (X) et l'estimateur des MCO (beta)
        # on a donc chaque colonne multiplié par un coefficient et on somme les colonnes entre elles pour obtenir une colonne qui est la prédiction du modèle complet

        partie_climat = complet - beta[-1] * X["tendance"]
        # niveau simulé sans la tendance (dernière colonne de X) : climat + constante (altitude), la tendance reste donc dans le résidu observé - simulé

        r2 = 1 - ((y - complet) ** 2).sum() / ((y - y.mean()) ** 2).sum() # calcul du R² du modèle complet, 
        # plus il est proche de 1 plus on explique bien les variations avec nos variables explicatives
        # on regarde s'il est meilleur que celui d'avant et si oui on actualise, le but est de trouver les parametres qui maximise R²
        if r2 > meilleur[0]:
            meilleur = (r2, k, partie_climat)
    return meilleur

In [18]:
def pente_sen(serie):
    """Pente de Sen saisonnière en cm/an : médiane des pentes entre les paires (i < j) du même mois calendaire.
    Deux mois différents ne sont jamais comparés : le cycle saisonnier ne peut pas fausser la pente."""
    s = serie.dropna()
    x = list((s.index - s.index[0]).days / 365.25)   #Temps en années depuis la première mesure
    y = list(s.values)
    mois = list(s.index.month)

    pentes = []
    n = len(x)
    for i in range(n):
        for j in range(i + 1, n):
            if mois[j] == mois[i]:   #Même mois d'années différentes, donc x[j] > x[i] : pas de division par zéro
                pentes.append((y[j] - y[i]) / (x[j] - x[i]))

    pentes.sort()
    m = len(pentes)
    if m % 2 == 1:
        mediane = pentes[m // 2]
    else:
        mediane = (pentes[m // 2 - 1] + pentes[m // 2]) / 2
    return mediane * 100   #m/an -> cm/an

In [19]:
lignes, partie_climat = {}, {}   # deux dicts indexés par station (clé = code_bss), remplis dans la même boucle donc dans le même ordre
                                 # lignes[station] = dict des indicateurs de la station -> deviendra une LIGNE de tendances
                                 # partie_climat[station] = Series mensuelle simulée sans la tendance -> deviendra une COLONNE de partie_climat
for station in niveaux_mensuels.columns:
    r2, k, partie_climat[station] = modele_climat(station)
    lignes[station] = {"r2": r2, "k_mois": k}   # il est plus rapide de remplir un dict puis de construire le df une seule fois que d'ajouter des lignes à un df

tendances = pd.DataFrame.from_dict(lignes, orient="index").rename_axis("code_bss")   # orient="index" : les clés deviennent l'index (une ligne par station) ; rename_axis nomme cet index
partie_climat = pd.DataFrame(partie_climat)   # orientation par défaut : les clés deviennent les colonnes, les séries sont alignées sur les dates
#Le retrait du climat (résidu et pentes corrigées) se fait en 4.5, une fois le modèle contrôlé

## 4.3 Contrôle : la mémoire k selon la profondeur et la roche

Si le modèle est physiquement cohérent, la mémoire k retenue doit être plus longue pour les nappes profondes et inertes (craie, captives)
que pour les nappes superficielles, qui réagissent vite à la pluie.

In [20]:
#Part des stations qui ont retenu chaque k, par classe de profondeur
controle = tendances.join(stations.drop_duplicates(subset="code_bss").set_index("code_bss")["classe_profondeur"])

def fig_memoire_profondeur():
    """Part des stations ayant retenu chaque mémoire k, par classe de profondeur."""
    pd.crosstab(controle["classe_profondeur"], controle["k_mois"], normalize="index").plot.bar(stacked=True, rot=0, figsize=(8, 5), xlabel="", ylabel="Part des stations", title="Mémoire k retenue selon la profondeur de l'ouvrage")
    plt.legend(title="k (mois)", bbox_to_anchor=(1, 1))   #Légende à droite pour ne pas cacher les barres
    plt.show()

if AFFICHER:
    fig_memoire_profondeur()

**Lecture.** Plus l'ouvrage est profond, plus la mémoire retenue est longue. La part des stations à k = 12 mois passe de 56 % (superficielles) à 18 % (intermédiaires) puis 14 % (profondes).
C'est un contrôle de cohérence, pas une preuve, car k ne prend que 3 valeurs et les R² de deux k voisins sont souvent proches.

## 4.4 Contrôle : le cycle saisonnier vient-il du climat ou des pompages ?

`partie_climat` contient le terme saison (sinus, cosinus), retiré du niveau en 4.5. C'est légitime seulement si le cycle annuel des nappes vient surtout du climat, et pas des pompages (plus forts en été).

In [21]:
#Chaque année, remontée d'hiver (octobre -> avril) et baisse d'été (avril -> octobre), face à la pluie efficace de la même saison
cycles = []
for station in niveaux_mensuels.columns:
    niveau, pe = niveaux_mensuels[station], pluie_eff_mensuelle[station]
    for an in range(2016, 2026):
        cycles.append({"code_bss": station,
                       "montee_hiver": niveau[pd.Timestamp(an, 4, 1)] - niveau[pd.Timestamp(an - 1, 10, 1)], "pe_hiver": pe[f"{an - 1}-10":f"{an}-03"].sum(),
                       "baisse_ete": niveau[pd.Timestamp(an, 10, 1)] - niveau[pd.Timestamp(an, 4, 1)], "pe_ete": pe[f"{an}-04":f"{an}-09"].sum()})
cycles = pd.DataFrame(cycles)

centre = cycles.groupby("code_bss").transform(lambda s: s - s.mean())   #Chaque station centrée : on ne compare que ses années entre elles

def fig_cycle_saisonnier():
    """Remontée d'hiver et baisse d'été face à la pluie efficace de la saison (écarts à la moyenne de chaque station)."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(centre["pe_hiver"], centre["montee_hiver"], s=10, alpha=0.4) # s = taille des points, alpha la transparence
    axes[1].scatter(centre["pe_ete"], centre["baisse_ete"], s=10, alpha=0.4)

    axes[0].set(title="Remontée d'hiver (octobre -> avril)", xlabel="Pluie efficace de la saison, écart à la moyenne de la station (mm)", ylabel="Variation du niveau, écart à la moyenne de la station (m)")
    axes[1].set(title="Baisse d'été (avril -> octobre)", xlabel="Pluie efficace de la saison, écart à la moyenne de la station (mm)", ylabel="Variation du niveau, écart à la moyenne de la station (m)")

    axes[0].axhline(0, color="dimgray", linewidth=0.8)  # ajout des axes en gris
    axes[1].axhline(0, color="dimgray", linewidth=0.8)
    axes[0].axvline(0, color="dimgray", linewidth=0.8)  
    axes[1].axvline(0, color="dimgray", linewidth=0.8)

    plt.tight_layout()
    plt.show()

if AFFICHER:
    fig_cycle_saisonnier()

**Lecture.** Dans les deux graphiques, les points montent de gauche à droite : un hiver plus pluvieux que d'habitude fait davantage remonter la nappe, un été plus pluvieux la fait moins baisser.
Le cycle change donc d'une année à l'autre avec la pluie, alors qu'un pompage régulier donnerait à peu près le même chaque année : il vient surtout du climat, et le retirer en 4.5 est justifié.
Réserve : les étés secs sont aussi ceux où l'on pompe le plus, ce qui peut accentuer la baisse d'été.

## 4.5 Retrait de la partie climatique

On retire à chaque station sa partie climatique (constante + pluie + saison) :
le résidu = observé − partie_climat garde la tendance $d \times t$ et tout ce que le climat n'explique pas.
On mesure ensuite la tendance avant (pente brute) et après (pente corrigée) ce retrait, par la pente de Sen saisonnière.

In [22]:
residu = niveaux_mensuels - partie_climat   #Mêmes mois en lignes, mêmes stations en colonnes
tendances["pente_brute_cm_an"] = niveaux_mensuels.apply(pente_sen)   #apply : pente_sen sur chaque colonne, donc chaque station
tendances["pente_corrigee_cm_an"] = residu.apply(pente_sen)
#Nappes captives (niveau moyen sous la mer) : le modèle suppose une recharge par la pluie locale, leur pente corrigée est à lire avec prudence
tendances["nappe_captive"] = niveaux_mensuels.mean() < 0
tendances.sort_values("pente_corrigee_cm_an").round(2)

,r2,k_mois,pente_brute_cm_an,pente_corrigee_cm_an,nappe_captive
code_bss,,,,,
00143C0079/F1,0.70,36,-42.24,-57.12,True
00134X0127/PZ1,0.81,24,-22.76,-35.85,True
BSS002PZXU/P,0.42,24,-12.92,-24.20,True
00281X0002/F1,0.64,36,1.05,-22.27,False
00202A0460/PZ12,0.56,36,-0.62,-22.08,False
00286X0405/PZST2,0.52,36,-11.30,-17.19,False
00384X0296/PZ4,0.68,24,57.62,-16.62,False
BSS004LJKU/X,0.76,24,-4.81,-15.81,False
00124X0023/F1,0.74,24,-3.80,-15.48,False


In [23]:
#Résumé des tendances avant et après retrait du climat
resume_tendances = pd.DataFrame({
    "pente brute (cm/an)": [tendances["pente_brute_cm_an"].median(), (tendances["pente_brute_cm_an"] < 0).sum(), (tendances["pente_brute_cm_an"] > 0).sum()],
    "pente corrigée (cm/an)": [tendances["pente_corrigee_cm_an"].median(), (tendances["pente_corrigee_cm_an"] < 0).sum(), (tendances["pente_corrigee_cm_an"] > 0).sum()],
}, index=["médiane", "stations en baisse", "stations en hausse"]).round(2)
print(resume_tendances)
print("Stations dont la pente corrigée est inférieure à la pente brute :", (tendances["pente_corrigee_cm_an"] < tendances["pente_brute_cm_an"]).sum(), "sur", len(tendances))

def fig_pentes_brute_corrigee():
    """Tendance de chaque station avant (x) et après (y) retrait du climat ; nappes captives en rouge."""
    plt.figure(figsize=(8, 7))
    couleurs = tendances["nappe_captive"].map({True: "darkred", False: "tab:blue"})
    plt.scatter(tendances["pente_brute_cm_an"], tendances["pente_corrigee_cm_an"], c=couleurs)
    bornes = [tendances[["pente_brute_cm_an", "pente_corrigee_cm_an"]].min().min(), tendances[["pente_brute_cm_an", "pente_corrigee_cm_an"]].max().max()]
    plt.plot(bornes, bornes, color="grey", linestyle="--", label="Pente corrigée = pente brute")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.legend()
    plt.title("Tendance de chaque station avant et après retrait du climat (rouge : nappes captives)")
    plt.xlabel("Pente brute (cm/an)")
    plt.ylabel("Pente corrigée du climat (cm/an)")
    plt.show()

if AFFICHER:
    fig_pentes_brute_corrigee()

                    pente brute (cm/an)  pente corrigée (cm/an)
médiane                           -0.62                    -4.6
stations en baisse                26.00                    41.0
stations en hausse                19.00                     4.0
Stations dont la pente corrigée est inférieure à la pente brute : 45 sur 45


**Lecture.** Toutes les stations sont sous la diagonale : pour chacune, la pente corrigée est plus basse que la pente brute.
Sur la période, la pluie a donc poussé les nappes vers le haut (années humides en fin de période, dont la recharge record de 2023-2024), ce qui masque une partie de la baisse dans le niveau observé.
Avant correction, les stations se partagent entre baisses (26) et hausses (19), avec une médiane proche de zéro (−0,6 cm/an).
Après correction, 41 stations sur 45 sont en baisse, avec une médiane de −4,6 cm/an : à pluie égale, la plupart des nappes baissent légèrement.
Les trois plus fortes baisses corrigées sont des nappes captives (Bondues, Nieppe, Quesnoy-sur-Deûle), pour lesquelles le modèle est à lire avec prudence.
Ce résidu mesure ce que la pluie n'explique pas, sans dire d'où il vient : il n'est pas, à ce stade, relié à la pression industrielle.

**Limite**

Le modèle sous-estime probablement l'effet humain (limite principale).
Les prélèvements ne sont pas répartis au hasard dans le temps : ils sont plus forts en été (irrigation, refroidissement, consommation)
et augmentent les années sèches. Or le modèle a justement deux termes qui varient de la même façon : la saison (sinus, cosinus) et la pluie efficace (faible les années sèches).
Tout ce que les prélèvements ont de saisonnier ou de lié à la sécheresse est donc compté comme « climat » et sort du résidu.

In [24]:
#Marqueur : cette étape est faite, les notebooks qui en dépendent ne la relanceront pas dans ce noyau
ETAPE_04 = True